# Decision tree dan random forest: bandingkan, jangan menebak

Kenali pohon keputusan dan ensemble sambil memilih model melalui cross-validation.

[Baca artikel lengkap](https://article.fikss.site/programming/ds-pohon-ensemble)

## Cara memakai
1. Masuk ke akun Google jika diminta.
2. Pilih **File → Save a copy in Drive / Simpan salinan di Drive** untuk menyimpan perubahan.
3. Jalankan sel dari atas ke bawah dengan tombol ▶ atau Shift+Enter. CPU sudah cukup.
4. Ubah kode contoh, jalankan ulang, lalu kerjakan latihan.

Data seluruhnya fiktif/sintetis. File sesi bersifat sementara; unduh hasil yang ingin disimpan. Jika sesi direset, jalankan ulang semua sel. Versi paket Colab dapat membuat hasil numerik sedikit berbeda dari artikel.


## 1. Siapkan pustaka
Sel ini memasang paket yang belum tersedia. Jika Colab meminta restart sesi setelah pemasangan, restart lalu jalankan lagi dari atas.


In [ ]:
%pip install -q "numpy>=1.26" "pandas>=2.2" "matplotlib>=3.8" "scikit-learn>=1.4" "scipy>=1.15" "openpyxl>=3.1.5"


## 2. Siapkan data latihan
Sel ini membuat dua CSV contoh dan helper pembersihan data pada folder sesi.


In [ ]:
from pathlib import Path
import sys
# Dataset fiktif dan helper disertakan; tidak perlu mengunduh ZIP.
berkas_latihan = {'datasets/monitoring.csv': 'waktu,perangkat,up,rtt_ms\n2026-10-01T00:00:00Z,BTS-A,1,20\n2026-10-01T00:01:00Z,BTS-A,1,25\n2026-10-01T00:02:00Z,BTS-A,0,\n2026-10-01T00:03:00Z,BTS-A,,\n2026-10-01T00:00:00Z,BTS-B,1,35\n2026-10-01T00:01:00Z,BTS-B,1,40\n2026-10-01T00:02:00Z,BTS-B,1,45\n2026-10-01T00:03:00Z,BTS-B,0,\n', 'datasets/penjualan.csv': 'id,tanggal,kota,produk,jumlah,harga_rupiah\nT001,2026-10-01,Surabaya,Router,2,500000\nT002,2026-10-01,Surabaya,Kabel,5,20000\nT003,2026-10-02,Malang,Router,1,500000\nT003,2026-10-02,Malang,Router,1,500000\nT004,2026-10-02,Malang,Kabel,-1,20000\nT005,2026-10-03,Surabaya,Switch,1,300000\nT006,2026-10-03,Surabaya,Kabel,,20000\nT007,2026-10-03,Malang,Kabel,3,20000\n', 'da04_kualitas.py': "from pathlib import Path\nimport numpy as np\nimport pandas as pd\n\ndef bersihkan_penjualan(raw):\n    wajib = {'id', 'tanggal', 'kota', 'produk', 'jumlah', 'harga_rupiah'}\n    if not wajib <= set(raw.columns):\n        raise ValueError('Kolom wajib belum lengkap.')\n    data = raw.drop_duplicates().copy()\n    duplikat_identik = len(raw) - len(data)\n    data['tanggal'] = pd.to_datetime(data['tanggal'], format='%Y-%m-%d', errors='coerce')\n    for col in ['id', 'kota', 'produk']:\n        data[col] = data[col].astype('string').str.strip()\n    for col in ['jumlah', 'harga_rupiah']:\n        data[col] = pd.to_numeric(data[col], errors='coerce')\n    valid = data['tanggal'].notna() & ~data['id'].duplicated(keep=False)\n    for col in ['id', 'kota', 'produk']:\n        valid &= data[col].notna() & data[col].ne('')\n    valid &= np.isfinite(data['jumlah']) & data['jumlah'].between(1, 10000) & data['jumlah'].mod(1).eq(0)\n    valid &= np.isfinite(data['harga_rupiah']) & data['harga_rupiah'].between(0, 1000000000) & data['harga_rupiah'].mod(1).eq(0)\n    valid = valid.fillna(False)\n    diterima = data.loc[valid].copy()\n    ditolak = data.loc[~valid].copy()\n    for col in ['jumlah', 'harga_rupiah']:\n        diterima[col] = diterima[col].astype('int64')\n    diterima['omzet_rupiah'] = diterima['jumlah'] * diterima['harga_rupiah']\n    return diterima, ditolak, duplikat_identik\n\nif __name__ == '__main__':\n    raw = pd.read_csv(Path(__file__).resolve().parent / 'datasets/penjualan.csv', dtype={'id':'string'})\n    baik, buruk, duplikat = bersihkan_penjualan(raw)\n    print('Mentah:', len(raw))\n    print('Duplikat identik:', duplikat)\n    print('Ditolak:', len(buruk))\n    print('Diterima:', len(baik))\n"}
for nama, isi in berkas_latihan.items():
    tujuan = Path.cwd() / nama
    tujuan.parent.mkdir(parents=True, exist_ok=True)
    tujuan.write_text(isi, encoding="utf-8")
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))
print("Dataset dan helper siap.")


## 3. Contoh: ds17_ensemble.py
Kode berikut sama dengan contoh di artikel. Baris `__file__` menyesuaikan lokasi CSV untuk notebook.


In [ ]:
__file__ = str(Path.cwd() / 'ds17_ensemble.py')
import numpy as np
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split,KFold,cross_val_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error
if __name__=='__main__':
    X,y=make_regression(n_samples=140,n_features=4,n_informative=3,noise=5,random_state=42)
    y=y+20*X[:,0]**2
    a,b,c,d=train_test_split(X,y,test_size=.2,random_state=42)
    calon={'baseline':DummyRegressor(strategy='median'),'tree':DecisionTreeRegressor(max_depth=3,random_state=42),
           'forest':RandomForestRegressor(n_estimators=50,max_depth=5,min_samples_leaf=3,random_state=42,n_jobs=1)}
    cv=KFold(n_splits=3,shuffle=True,random_state=42)
    skor={n:-cross_val_score(m,a,c,cv=cv,scoring='neg_mean_absolute_error',n_jobs=1).mean() for n,m in calon.items()}
    terbaik=min(skor,key=skor.get)
    print('MAE CV:',{n:round(float(v),2) for n,v in skor.items()})
    model=clone(calon[terbaik]).fit(a,c)
    print('Terpilih dari CV:',terbaik)
    print('MAE test akhir:',round(mean_absolute_error(d,model.predict(b)),2))


### Acuan keluaran
```text
MAE CV: {'baseline': 77.61, 'tree': 51.69, 'forest': 30.56}
Terpilih dari CV: forest
MAE test akhir: 18.05
```


## 4. Lihat grafik dan hasil
Grafik yang dibuat contoh ditampilkan di bawah. File CSV/PNG tersedia di panel Files Colab untuk diunduh.


In [ ]:
from IPython.display import display, Image
folder_hasil = Path("hasil-latihan")
if folder_hasil.exists():
    for hasil in sorted(folder_hasil.iterdir()):
        if hasil.is_file():
            print(hasil)
            if hasil.suffix.lower() == ".png":
                display(Image(filename=str(hasil)))


## 5. Coba sendiri
1. Ubah max_depth pada salinan kandidat dan amati trade-off validasi.
2. Bandingkan kebutuhan waktu/ukuran model tanpa menganggap angka benchmark satu mesin sebagai hukum umum.

In [ ]:
# Tulis percobaan dan jawabanmu di sini.


## Cek pemahaman
Mengapa model di-fit ulang pada seluruh train setelah dipilih dari CV?

Kembali ke artikel untuk tes pemahaman dan pembahasan.

**Perhatikan:** Jangan memilih algoritma karena namanya lebih canggih. Data, validasi, baseline, serta biaya operasi menentukan keputusan yang masuk akal.
